In [0]:
import json
import os

import requests
from pyspark.sql import functions as F

api_url = "https://www.geoboundaries.org/api/current/gbOpen/MAR/ADM1/"
meta = requests.get(api_url, timeout=60).json()
file_url = meta["simplifiedGeometryGeoJSON"]
version = file_url.split("/raw/")[1].split("/")[0]

target_dir = f"/Volumes/workspace/bank_lakehouse/landing/regions/source_version={version}"
path = f"{target_dir}/regions_adm1.geojson"

if os.path.exists(path):
    print("Version déjà présente, téléchargement ignoré.")
else:
    response = requests.get(file_url, timeout=120)
    response.raise_for_status()
    if len(response.json()["features"]) != 12:
        raise ValueError("On attend 12 régions : fichier refusé.")
    os.makedirs(target_dir, exist_ok=True)
    with open(path, "w", encoding="utf-8") as file:
        file.write(response.text)
    with open(f"{target_dir}/metadata.json", "w", encoding="utf-8") as file:
        json.dump(meta, file, ensure_ascii=False, indent=2)
    print("Fichier enregistré :", path)

print("Version de la source :", version)

In [0]:
with open(path, encoding="utf-8") as file:
    geojson = json.load(file)

rows = [
    (
        feature["properties"].get("shapeName"),
        feature["properties"].get("shapeISO"),
        feature["properties"].get("shapeID"),
        json.dumps(feature["geometry"]),
    )
    for feature in geojson["features"]
]

bronze_regions = (
    spark.createDataFrame(rows, "shape_name string, shape_iso string, shape_id string, geometry_geojson string")
    .withColumn("_source_version", F.lit(version))
    .withColumn("_source_file", F.lit(path))
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze_regions.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.bank_lakehouse.bronze_regions")

display(spark.table("workspace.bank_lakehouse.bronze_regions")
        .select("shape_name", "shape_iso", F.length("geometry_geojson").alias("taille_geometrie")))